## 🎯 Learning Objectives
* Understand the critical role of regression testing in maintaining LLM performance and stability in production.
* Learn how to design and implement A/B evaluation strategies for comparing different LLM versions or prompt engineering approaches.
* Identify appropriate metrics and methodologies for both regression and A/B testing in an LLMOps context.
* Recognize the trade-offs and best practices associated with deploying and monitoring LLM changes using these evaluation techniques.


## Lesson OPS01-L07: Regression Testing and A/B Evaluation in LLMOps

In the dynamic world of Large Language Models (LLMs), continuous iteration is key to improvement. However, every change—whether it's a new model version, a tweaked prompt, an updated RAG strategy, or even a minor code refactor—carries the risk of introducing unintended side effects or degrading existing performance. This is where **Regression Testing** and **A/B Evaluation** become indispensable tools in an LLMOps pipeline.

### 1. Regression Testing: Ensuring Stability with Every Change

Imagine you've just optimized your LLM's prompt to handle a new type of customer query more effectively. Great! But what if this change inadvertently makes the model worse at answering common, well-established queries? This is the problem regression testing solves.

**What is it?**
Regression testing in LLMOps is the process of re-running a set of predefined, 'golden' test cases against a new version of your LLM system (model, prompt, RAG, etc.) to ensure that recent changes haven't introduced new bugs or degraded previously working functionalities. It's like a safety net, catching regressions before they impact users.

**Why is it crucial for LLMs?**
LLMs are complex, non-deterministic systems. A small change can have far-reaching, unpredictable consequences. Regression testing helps:
*   **Maintain Quality:** Ensures that core functionalities remain robust.
*   **Prevent Production Incidents:** Catches performance degradations before deployment.
*   **Build Confidence:** Allows engineers to iterate faster with a safety net.
*   **Track Performance Over Time:** Provides a historical record of how changes impact performance.

**How it works (Simplified):**
1.  **Define Golden Test Cases:** Create a curated set of input-expected output pairs that represent critical functionalities or common user queries. These are your 'ground truth'.
2.  **Establish Baselines:** Run your current production LLM system against these golden test cases and record its performance (e.g., accuracy, specific metric scores). This is your 'baseline'.
3.  **Introduce Changes:** Develop your new LLM version (e.g., new prompt, fine-tuned model).
4.  **Run Regression Tests:** Execute the new LLM version against the *same* golden test cases.
5.  **Compare and Alert:** Compare the new performance metrics against the baseline. If performance drops below a predefined threshold for any test case, it signals a regression, and the change should be investigated or rolled back.

### 2. A/B Evaluation: The Scientific Approach to Improvement

Once you've ensured your new LLM version hasn't regressed, how do you know if it's actually *better*? This is where A/B evaluation comes in.

**What is it?**
A/B evaluation (or A/B testing) is a controlled experiment where two or more versions of an LLM system (e.g., 'A' and 'B') are exposed to different segments of your user base or a controlled set of evaluation data. The goal is to determine which version performs better against specific, measurable objectives (e.g., user satisfaction, task completion rate, response quality).

**Why is it crucial for LLMs?**
LLM improvements are often subtle and can be subjective. A/B testing provides a data-driven way to:
*   **Validate Hypotheses:** Scientifically prove if a new model, prompt, or RAG strategy is genuinely better.
*   **Optimize User Experience:** Directly measure the impact of changes on real users or realistic scenarios.
*   **Reduce Risk:** Gradually roll out changes to a small segment before full deployment.
*   **Inform Decision Making:** Provide concrete data to support deployment decisions.

**How it works (Simplified):**
1.  **Define Hypothesis:** State what you expect to improve (e.g., "Prompt B will lead to higher user satisfaction than Prompt A").
2.  **Identify Metrics:** Choose quantifiable metrics to measure success (e.g., click-through rate, explicit user feedback, task success rate, automated quality scores).
3.  **Create Variants:** Prepare your 'A' version (control, often current production) and 'B' version (the new change you want to test).
4.  **Traffic Splitting (or Data Splitting):** Direct a portion of incoming requests (or evaluation data) to version A and another portion to version B. Ensure the split is random and unbiased.
5.  **Collect Data:** Monitor and collect data on the chosen metrics for both versions over a statistically significant period.
6.  **Analyze Results:** Compare the performance of A and B. Use statistical methods to determine if any observed differences are significant or merely due to chance.
7.  **Decide:** Based on the analysis, decide whether to fully deploy B, iterate further, or stick with A.

Both regression testing and A/B evaluation are cornerstones of a robust LLMOps strategy, enabling AI engineers and DevOps specialists to confidently build, deploy, and continuously improve LLM-powered applications.


In [ ]:
import random
import numpy as np
from collections import defaultdict

# --- Mock LLM and Evaluation Utilities ---

class MockLLM:
    """A simplified mock LLM for demonstration purposes."""
    def __init__(self, name="MockLLM", behavior_strategy=None):
        self.name = name
        self.behavior_strategy = behavior_strategy if behavior_strategy else self._default_behavior

    def _default_behavior(self, prompt):
        # Simple keyword-based response simulation
        prompt_lower = prompt.lower()
        if "hello" in prompt_lower or "hi" in prompt_lower:
            return f"Hello there! How can {self.name} assist you?"
        elif "weather" in prompt_lower:
            return f"The weather is sunny and 25°C today, according to {self.name}."
        elif "recommend" in prompt_lower and "book" in prompt_lower:
            return f"I recommend 'Project Hail Mary' by Andy Weir, says {self.name}."
        elif "summarize" in prompt_lower:
            return f"Here's a summary of your request: '{prompt[:30]}...' - {self.name}"
        else:
            return f"I'm not sure how to respond to '{prompt}', but {self.name} is learning!"

    def generate(self, prompt):
        return self.behavior_strategy(prompt)


def exact_match_score(predicted, expected):
    """Calculates exact match accuracy."""
    return 1 if predicted.strip().lower() == expected.strip().lower() else 0

def keyword_match_score(predicted, expected_keywords):
    """Calculates a score based on how many expected keywords are in the prediction."""
    predicted_lower = predicted.lower()
    score = 0
    for keyword in expected_keywords:
        if keyword.lower() in predicted_lower:
            score += 1
    return score / len(expected_keywords) if expected_keywords else 0


# --- Regression Testing Example ---

print("--- LLM Regression Testing Example ---")

# 1. Define Golden Test Cases (Input, Expected Output)
# In a real scenario, these would be carefully curated and extensive.
regression_golden_dataset = [
    {"input": "Say hello", "expected": "hello there! how can mockllm assist you?"},
    {"input": "What's the weather like?", "expected": "the weather is sunny and 25°c today, according to mockllm."},
    {"input": "Recommend a good book", "expected": "i recommend 'project hail mary' by andy weir, says mockllm."},
    {"input": "Summarize this text: The quick brown fox jumps over the lazy dog.", "expected": "here's a summary of your request: 'the quick brown fox jumps...' - mockllm"}
]

# 2. Baseline LLM (e.g., current production version)
llm_baseline = MockLLM(name="BaselineLLM")

# 3. New LLM (e.g., with a prompt change or model update)
# Let's simulate a 'bug' in the new LLM's weather response for demonstration
def new_llm_behavior(prompt):
    prompt_lower = prompt.lower()
    if "weather" in prompt_lower:
        return "I cannot provide weather information at this time. Please try again later." # Simulating a regression
    return llm_baseline.generate(prompt) # Fallback to baseline for other queries

llm_new_version = MockLLM(name="NewLLM", behavior_strategy=new_llm_behavior)

# 4. Run Regression Tests and Evaluate

def run_regression_test(llm, dataset):
    results = []
    for i, test_case in enumerate(dataset):
        input_text = test_case["input"]
        expected_output = test_case["expected"]
        predicted_output = llm.generate(input_text)
        score = exact_match_score(predicted_output, expected_output)
        results.append({
            "test_id": i + 1,
            "input": input_text,
            "expected": expected_output,
            "predicted": predicted_output,
            "score": score,
            "passed": score == 1
        })
    return results

print("\nRunning Baseline LLM Regression Tests...")
baseline_results = run_regression_test(llm_baseline, regression_golden_dataset)
for res in baseline_results:
    print(f"  Test {res['test_id']}: Input='{res['input']}' | Passed: {res['passed']} (Score: {res['score']:.2f})")

print("\nRunning New LLM Version Regression Tests...")
new_version_results = run_regression_test(llm_new_version, regression_golden_dataset)
for res in new_version_results:
    print(f"  Test {res['test_id']}: Input='{res['input']}' | Passed: {res['passed']} (Score: {res['score']:.2f})")

# 5. Compare and Identify Regressions
print("\n--- Regression Test Comparison ---")
regressions_found = False
for i in range(len(regression_golden_dataset)):
    baseline_pass = baseline_results[i]["passed"]
    new_pass = new_version_results[i]["passed"]
    if baseline_pass and not new_pass:
        regressions_found = True
        print(f"!!! REGRESSION DETECTED !!! Test {i+1}: '{regression_golden_dataset[i]['input']}'")
        print(f"    Baseline Output: '{baseline_results[i]['predicted']}'")
        print(f"    New Version Output: '{new_version_results[i]['predicted']}'")
        print(f"    Expected Output: '{regression_golden_dataset[i]['expected']}'")

if not regressions_found:
    print("No regressions detected. New version maintains baseline performance on golden dataset.")
else:
    print("Regressions detected. Investigate changes in the new LLM version.")


# --- A/B Evaluation Example ---

print("\n\n--- LLM A/B Evaluation Example ---")

# Scenario: We want to compare two different prompt strategies for a customer support bot.
# Version A: Standard, direct prompt.
# Version B: More polite, empathetic prompt, hoping for better user satisfaction.

# Mock LLM with a base behavior
base_llm_for_ab = MockLLM(name="BaseSupportLLM")

# Prompt Strategy A
def prompt_strategy_A(query):
    return f"Answer the following customer query directly: {query}"

# Prompt Strategy B (more empathetic)
def prompt_strategy_B(query):
    return f"As a helpful and empathetic customer support agent, please respond to the following query: {query}. Ensure your tone is friendly and understanding."

# Simulate LLM responses based on strategies
class LLMVariant:
    def __init__(self, name, prompt_strategy, base_llm):
        self.name = name
        self.prompt_strategy = prompt_strategy
        self.base_llm = base_llm

    def generate(self, query):
        processed_prompt = self.prompt_strategy(query)
        # For simplicity, our mock LLM's response is still keyword-based
        # but in a real scenario, the prompt strategy would influence the actual LLM's output.
        # Here, we'll simulate a slight difference in output based on the strategy name.
        base_response = self.base_llm.generate(processed_prompt)
        if self.name == "Variant B": # Simulate 'better' response for B
            return base_response.replace("not sure", "happy to help you further")
        return base_response

llm_variant_A = LLMVariant("Variant A", prompt_strategy_A, base_llm_for_ab)
llm_variant_B = LLMVariant("Variant B", prompt_strategy_B, base_llm_for_ab)

# Define common test queries for A/B comparison
ab_test_queries = [
    "My order hasn't arrived yet.",
    "How do I reset my password?",
    "I want to cancel my subscription.",
    "Can you tell me about your refund policy?",
    "My product is broken."
]

# Simulate user feedback/evaluation (e.g., human rating, or automated sentiment analysis)
# For this example, we'll use a simplified 'quality score' based on keywords.

def evaluate_response_quality(response, query_type):
    response_lower = response.lower()
    score = 0
    if "happy to help" in response_lower or "assist you" in response_lower:
        score += 0.5 # Empathetic tone
    if "order" in query_type and "tracking" in response_lower:
        score += 1
    if "password" in query_type and "reset link" in response_lower:
        score += 1
    if "cancel" in query_type and "subscription" in response_lower and "process" in response_lower:
        score += 1
    if "refund" in query_type and "policy" in response_lower and "details" in response_lower:
        score += 1
    if "broken" in query_type and "troubleshoot" in response_lower:
        score += 1
    return min(score, 2) # Max score of 2 for simplicity

# Run A/B Test
results_A = []
results_B = []

print("\nRunning A/B Test...")
for query in ab_test_queries:
    # Simulate traffic splitting - here, all queries go to both for direct comparison
    # In a real A/B test, users would be randomly assigned to A or B.
    response_A = llm_variant_A.generate(query)
    response_B = llm_variant_B.generate(query)

    quality_A = evaluate_response_quality(response_A, query.lower())
    quality_B = evaluate_response_quality(response_B, query.lower())

    results_A.append(quality_A)
    results_B.append(quality_B)

    print(f"  Query: '{query}'")
    print(f"    Variant A Score: {quality_A:.2f} | Response: '{response_A[:70]}...' ")
    print(f"    Variant B Score: {quality_B:.2f} | Response: '{response_B[:70]}...' ")

# Analyze Results
mean_score_A = np.mean(results_A)
mean_score_B = np.mean(results_B)

print("\n--- A/B Test Results ---")
print(f"Average Quality Score for Variant A: {mean_score_A:.2f}")
print(f"Average Quality Score for Variant B: {mean_score_B:.2f}")

if mean_score_B > mean_score_A:
    print("Variant B (Empathetic Prompt) performed better on average. Consider deploying it!")
elif mean_score_A > mean_score_B:
    print("Variant A (Direct Prompt) performed better on average. Stick with it or refine B.")
else:
    print("Both variants performed similarly. Further testing or refinement may be needed.")

# In a real scenario, you'd perform statistical significance tests (e.g., t-test, chi-squared) here.
# For example, using scipy.stats:
# from scipy import stats
# t_stat, p_value = stats.ttest_ind(results_A, results_B)
# print(f"T-statistic: {t_stat:.2f}, P-value: {p_value:.3f}")
# if p_value < 0.05: print("The difference is statistically significant.")


### Interpreting the Code Output and Practical Considerations

#### Regression Testing Output Interpretation

The regression testing example clearly demonstrates how a seemingly small change (like our simulated 'bug' in the `NewLLM`'s weather response) can lead to a regression. The output highlights:

*   **Baseline Performance:** Shows that the `BaselineLLM` successfully passed all golden tests.
*   **New Version Performance:** Reveals that `NewLLM` failed the weather-related test, indicating a regression.
*   **Regression Detection:** The final comparison explicitly flags the regression, showing the input, expected output, and the erroneous new output. This immediate feedback is crucial for developers to identify and fix issues before deployment.

**Performance Trade-offs & Use Cases:**

*   **Cost of Golden Datasets:** Creating and maintaining high-quality, diverse golden datasets is labor-intensive but essential. These datasets should cover critical functionalities, edge cases, and common user queries. Tools like `Argilla` or `Snorkel AI` can assist in data labeling and management.
*   **Computational Cost:** Re-running extensive test suites can be computationally expensive, especially with large models. Strategies include running full suites nightly, smaller smoke tests on every commit, and parallelizing evaluations.
*   **Deterministic vs. Non-Deterministic LLMs:** Exact match is simple but often too strict for LLMs. More sophisticated metrics (semantic similarity using embedding models, ROUGE, BLEU, custom rubric-based evaluations) are often needed. For non-deterministic models, you might run tests multiple times and evaluate the *distribution* of scores.
*   **Integration:** Regression tests should be integrated into your CI/CD pipeline (e.g., GitHub Actions, GitLab CI, Jenkins) to automatically block deployments if regressions are detected.

#### A/B Evaluation Output Interpretation

The A/B evaluation example simulates comparing two prompt strategies. The output shows:

*   **Individual Query Scores:** For each query, you see the simulated quality score and a snippet of the response for both Variant A and Variant B.
*   **Average Scores:** The final average quality scores for each variant provide a high-level comparison.
*   **Decision Guidance:** Based on the average scores, the example provides a preliminary recommendation. In our case, Variant B (the empathetic prompt) showed a slightly better average score due to the simulated 'better' response for certain conditions.

**Performance Trade-offs & Use Cases:**

*   **Statistical Significance:** The most critical aspect of A/B testing. Our example uses simple averages, but in reality, you *must* use statistical tests (like t-tests for continuous data, chi-squared for categorical data) to determine if the observed difference is truly significant or just random variation. A `p-value` below a threshold (e.g., 0.05) indicates significance.
*   **Duration and Traffic:** A/B tests need to run long enough and expose enough traffic to gather statistically significant data. This can take days or weeks, depending on your traffic volume and the magnitude of the expected effect. Tools like `Optimizely` or `Google Optimize` (or custom solutions built on cloud platforms like `Vertex AI`'s A/B testing features) help manage traffic splitting and data collection.
*   **Metric Selection:** Choosing the right metrics is paramount. These could be direct user feedback (thumbs up/down), task completion rates, conversion rates, time spent on page, or automated metrics like sentiment, coherence, or factuality scores.
*   **Ethical Considerations:** Be mindful of potential negative user experiences during A/B tests, especially if one variant performs significantly worse. Consider canary deployments or dark launches before full A/B tests.
*   **Experiment Tracking:** Tools like `MLflow` or `Weights & Biases` are invaluable for tracking experiment configurations, metrics, and results for different A/B test variants.

Both regression testing and A/B evaluation are continuous processes. They are not one-off tasks but integral parts of the LLMOps lifecycle, ensuring that your LLM applications remain robust, performant, and continuously improve in a data-driven manner.


### Resources

*   **LangChain Evaluation:** [https://python.langchain.com/docs/guides/evaluation/](https://python.langchain.com/docs/guides/evaluation/)
*   **LlamaIndex Evaluation:** [https://docs.llamaindex.ai/en/stable/module_guides/evaluating/root.html](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/root.html)
*   **Hugging Face Evaluate Library:** [https://huggingface.co/docs/evaluate/index](https://huggingface.co/docs/evaluate/index)
*   **MLflow for Experiment Tracking:** [https://mlflow.org/docs/latest/llms/index.html](https://mlflow.org/docs/latest/llms/index.html)
*   **Google Cloud Vertex AI A/B Testing:** [https://cloud.google.com/vertex-ai/docs/model-monitoring/ab-testing](https://cloud.google.com/vertex-ai/docs/model-monitoring/ab-testing)
*   **General A/B Testing Best Practices:** [https://www.optimizely.com/optimization-glossary/ab-testing/](https://www.optimizely.com/optimization-glossary/ab-testing/)
*   **Argilla (Data Labeling & Monitoring):** [https://argilla.io/](https://argilla.io/)
